##Creación de Archivo de Datos DATASET COSTOS HISTORICOS

In [1]:
import numpy as np
import pandas as pd

In [2]:
# 1. Configurar la semilla aleatoria para que los datos sean reproducibles
np.random.seed(42) # Con el número 42, nos aseguramos que los datos simulados sean siempre los mismos cada vez que se abra el archivo.

#Generar el vector del tiempo (36 meses) Representa la línea de tiempo de 3 años histórico mes a mes.
meses = np.arange(1, 37)

In [3]:
# 2. Simular las variables predictoras (Features) con lógica de negocio
gastos_publicidad = np.random.uniform(200000, 600000, 36) # Simula que la empresa invierte en publicidad un monto variable que oscila $200.000 y $600.000
gastos_publicidad[35] = 500000 + np.random.normal(0, 5000) # Mes 36 aprox 500,000 para acercarnos a ese monto en el ultimo mes


In [4]:
# 3. Simular Costo Variable Unitario (Crece con inflación para llegar a $3,000 en el mes 36)
costo_variable_unitario = 840 + (meses * 60) + np.random.normal(0, 25, 36)
costo_variable_unitario[35] = 3000 + np.random.normal(0, 10) # Mes 36 aprox 3,000

In [5]:
# 4. Simular Precio de Venta (Acompaña la escala comercial para tocar los $7,000 en el mes 36)
precio_venta_docena = 2000 + (meses * 138) + np.random.normal(0, 40, 36)
precio_venta_docena[35] = 7000 + np.random.normal(0, 15) # Mes 36 aprox 7,000

In [6]:
# 5. Simular Demanda (Crecimiento fuerte de la empresa mes a mes hasta llegar a 50.000 docenas)
estacionalidad = np.sin(meses * (2 * np.pi / 12)) * 400 #Demanda con Picos en Inverno y bajas en Verano.
error_humano = np.random.normal(0, 150, 36)
# La ecuación incluye un multiplicador por 'meses' para simular la expansión del negocio en el tiempo
docenas_demandadas = 1200 + (meses * 1350) + (gastos_publicidad * 0.001) + estacionalidad + error_humano
docenas_demandadas[35] = 50000 + np.random.normal(0, 80) # Mes 36 aprox 50,000

In [7]:
# 6. Estructurar en el DataFrame y aplicar redondeo estricto a 2 decimales
df_historico_costos = pd.DataFrame({
    'Mes_Cronologico': meses,
    'Gastos_Publicidad': gastos_publicidad,
    'Costo_Variable_Unitario': costo_variable_unitario,
    'Precio_Venta_Docena': precio_venta_docena,
    'Docenas_Demandadas': docenas_demandadas
}).round(2)

In [8]:
# 7. Simular Costos Fijos Totales mes a mes (Estructura de fábrica + Administración)
# Arrancan en aprox $1.500.000 el mes 1 y escalan hasta aprox $12.000.000 el mes 36 por el crecimiento de la empresa
error_fijos = np.random.normal(0, 50000, 36)
df_historico_costos['Costos_Fijos_Totales'] = (1200000 + (df_historico_costos['Mes_Cronologico'] * 300000) + error_fijos).round(2) #Crea la columna de costos fijos mensuales.
# Forzar el mes 36 a un valor coherente de estructura para soportar 50.000 docenas
df_historico_costos.loc[df_historico_costos['Mes_Cronologico'] == 36, 'Costos_Fijos_Totales'] = 12000000.00

In [9]:
# 8. Calcular el Margen de Contribución Unitario de cada mes Precio de Venta - Costo Variable Unitario
margen_contribucion_unitario = df_historico_costos['Precio_Venta_Docena'] - df_historico_costos['Costo_Variable_Unitario']

In [10]:
# 9. Calcular la Cantidad de Equilibrio histórica aplicando la fórmula financiera Costos Fijos/Margen De Contribucion
df_historico_costos['Cantidad_Equilibrio'] = (df_historico_costos['Costos_Fijos_Totales'] / margen_contribucion_unitario).round(2)

In [11]:
# 10. Crear la columna Lag temporal (la demanda del mes anterior)
df_historico_costos['Demanda_Mes_Anterior'] = df_historico_costos['Docenas_Demandadas'].shift(1)

# 11. Como el shift vuelve a dejar una celda vacía en la primera fila de la tabla actual, la rellenamos con un promedio
df_historico_costos['Demanda_Mes_Anterior'] = df_historico_costos['Demanda_Mes_Anterior'].bfill()






In [12]:
#12 Definir el nuevo orden de las columnas dejando las demandas juntas
columnas_ordenadas = [
    'Mes_Cronologico',
    'Gastos_Publicidad',
    'Costo_Variable_Unitario',
    'Precio_Venta_Docena',
    'Costos_Fijos_Totales',
    'Cantidad_Equilibrio',
    'Docenas_Demandadas',
    'Demanda_Mes_Anterior',  # Colocada al lado
        # de la Target docenas demandadas
]

# Aplicar el orden al DataFrame existente
df_historico_costos = df_historico_costos[columnas_ordenadas]



####La idea fue establecer los datos del MES 36, con números aproximados y relacionados a la automatización realizada anteriormente, para que sirva como base y luego se simulen los datos de los MESES 1 al 35.

In [13]:
df_historico_costos.head(40
)

,Mes_Cronologico,Gastos_Publicidad,Costo_Variable_Unitario,Precio_Venta_Docena,Costos_Fijos_Totales,Cantidad_Equilibrio,Docenas_Demandadas,Demanda_Mes_Anterior
0,1,349816.05,892.71,2135.12,1511373.00,1216.48,2979.47,2979.47
1,2,580285.72,944.96,2316.14,1865357.14,1360.40,4802.50,2979.47
2,3,492797.58,1066.31,2428.47,2019625.84,1482.66,6203.41,4802.50
3,4,439463.39,1079.66,2526.20,2409231.69,1665.51,7668.80,6203.41
4,5,262407.46,1113.56,2704.46,2712994.14,1705.32,8438.59,7668.80
5,6,262397.81,1220.56,2889.52,3039091.14,1820.95,9601.03,8438.59
6,7,223233.44,1229.48,2964.57,3238152.46,1866.27,10662.07,9601.03
7,8,546470.46,1325.22,3166.59,3533977.17,1919.21,11912.24,10662.07
8,9,440446.00,1331.01,3137.21,3926097.08,2173.68,13386.47,11912.24
9,10,483229.03,1406.80,3412.88,4214849.23,2101.04,14845.85,13386.47


In [14]:
# Guardar el DataFrame en un archivo CSV
df_historico_costos.to_csv('historico_costos_demanda.csv', index=False)

# Para descargarlo directo a la computadora, con estas dos líneas:
from google.colab import files
files.download('historico_costos_demanda.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###***NIETO JORGE OSCAR***